In [ ]:
import random
from datasets import load_dataset, Dataset, IterableDataset
from typing import List, Dict

# ===========================================================================
# 💡 프로젝트 개요: 군사 드로잉 묘사 프롬프트 분석기 🚀
# ---------------------------------------------------------------------------
# [데이터셋 이름] Falah/military_drawing_descriptions
# [의미] 군사 드로잉이나 스케치를 묘사하는 영어 프롬프트(묘사문) 모음입니다.
# [목표] 이 데이터셋은 이미지 생성 AI(예: Midjourney, DALL-E)에 사용할 '프롬프트'를 학습하는 데 최적화되어 있습니다.
#       우리는 이 데이터를 사용하여, 다양한 프롬프트들이 어떤 공통적인 특징(키워드, 스타일)을 가지는지 초보자도 쉽게 파악해 봅시다!
# ---------------------------------------------------------------------------

# 사용자 설정 변수
DATASET_NAME = "Falah/military_drawing_descriptions"
TARGET_SPLIT = "train"
SAMPLE_COUNT = 5 # ⚠️ 데이터셋이 크므로, 일단 처음 5개만 분석해볼 거예요!

print("✨ 환영합니다! AI 파이프라인의 첫 단추를 꿰어 볼 시간입니다. ✨")
print("우리는 텍스트 데이터가 어떻게 AI의 '지식'으로 변하는지 체험할 거예요.")
print("-----------------------------------------------------------------------")


# ---------------------------------------------------------------------------
# 📦 1단계: 데이터 로드 및 안정화 (스트리밍 모드 처리)
# ---------------------------------------------------------------------------
dataset = None
try:
    # 🚀 스트리밍(streaming=True) 모드로 빠르게 데이터 로드를 시도해봅니다.
    print(f"⚙️ {DATASET_NAME} 데이터셋을 스트리밍 모드로 로드합니다...")
    dataset = load_dataset(DATASET_NAME, split=TARGET_SPLIT, streaming=True)
    print("✅ 성공! 스트리밍 방식으로 데이터셋을 메모리에 올렸습니다. 정말 빠르죠?")

except Exception as e:
    # 😔 만약 스트리밍 로드에 실패한다면, 일반 다운로드 방식으로 전환합니다.
    print(f"\n⚠️ 스트리밍 로드 실패 (오류: {e}). 일반 다운로드 방식으로 {TARGET_SPLIT} 스플릿을 다운로드합니다...")
    try:
        dataset = load_dataset(DATASET_NAME, split=TARGET_SPLIT, streaming=False)
        print("✅ 전환 성공! 일반 다운로드 방식으로 데이터를 로드했습니다.")
    except Exception as e_fallback:
        print(f"\n❌ 치명적인 오류 발생: 데이터셋 로드에 완전히 실패했습니다. {e_fallback}")
        exit()


# ---------------------------------------------------------------------------
# ⚙️ 2단계: 샘플링 및 반복자 설정 (핵심 패턴 적용)
# ---------------------------------------------------------------------------

print("\n✨ 이제 데이터 분석을 위한 '탐험가 도구'를 설정할 차례입니다.")

# 📝 스트리밍 데이터셋(IterableDataset)인지 확인하고, 적절한 반복자(iterator)를 생성합니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)인 경우입니다.
    print("📊 (감지) 스트리밍 모드입니다. .take()를 사용하여 상위 샘플만 가져와야 해요.")
    # next()를 쓰기 위해 iter()로 감싸주고, take(K)를 사용합니다.
    sample_iterator = iter(dataset.take(SAMPLE_COUNT))
else:
    # 일반 데이터셋 (Dataset)인 경우입니다.
    print("📦 (감지) 일반 메모리 데이터셋입니다. list(dataset.take(K))를 사용하여 샘플을 가져와요.")
    # 일반 데이터셋은 list()로 변환해도 안전합니다.
    sampled_dataset = list(dataset.take(SAMPLE_COUNT))
    sample_iterator = iter(sampled_dataset)


# ---------------------------------------------------------------------------
# 🧩 3단계: 초보자용 창의 실습 - 프롬프트 키워드 분석 (NLP/분석 실습)
# ---------------------------------------------------------------------------

print("\n==========================================================")
print("✨ 🎨 실습 시작: 프롬프트에서 핵심 키워드 추출하기 (Prompt Engineering 시뮬레이션)")
print("==========================================================")

# 🚀 키워드 분석을 위한 함수 정의 (사용자 정의 로직)
def extract_key_features(prompt: str) -> List[str]:
    """
    주어진 프롬프트에서 '군사적', '드로잉', '분위기'와 관련된 키워드를 추출합니다.
    (매우 간단한 시뮬레이션입니다. 실제 NLP 모델이 필요해요!)
    """
    keywords = []
    lowered_prompt = prompt.lower()

    # 1. 주제 키워드 체크 (군사/전투)
    if any(word in lowered_prompt for word in ["battle", "soldier", "army", "weapon", "tank"]):
        keywords.append("🛡️ Military")
    
    # 2. 스타일 키워드 체크 (드로잉/묘사)
    if "drawing" in lowered_prompt or "sketch" in lowered_prompt or "pencil" in lowered_prompt:
        keywords.append("✏️ Sketch Style")

    # 3. 분위기/기술 키워드 체크 (조명/시점)
    if "dramatic" in lowered_prompt or "shadow" in lowered_prompt or "sepia" in lowered_prompt:
        keywords.append("🎭 Moody Lighting")
        
    # 4. 기본적인 키워드 추출 (빈도수 계산 시뮬레이션)
    potential_words = ["detailed", "realistic", "epic", "smoke", "action"]
    for word in potential_words:
        if word in lowered_prompt:
            keywords.append(word.upper())
            
    return list(set(keywords)) # 중복 제거 후 반환

print(f"\n🔎 상위 {SAMPLE_COUNT}개 샘플을 순회하며 분석을 진행합니다...")

# 💖 반복자를 이용해 샘플을 순회합니다. (next() 패턴 사용)
sample_index = 0
for i in range(SAMPLE_COUNT):
    try:
        sample_data = next(sample_iterator)
        sample_index += 1
        
        # 데이터셋에서 'prompts' 필드를 가져옵니다.
        prompt_text = sample_data['prompts']
        
        print(f"\n[🔍 Sample {sample_index} / {SAMPLE_COUNT}]")
        print(f"   ➡️ 원본 프롬프트: {prompt_text[:70]}...") # 너무 길면 자릅니다.
        
        # 🔑 키워드 추출 함수를 호출합니다.
        keywords = extract_key_features(prompt_text)
        
        # ✨ 분석 결과 출력
        if keywords:
            print("   ✨ [분석 결과] 예상되는 핵심 키워드:")
            for kw in keywords:
                print(f"   🌟 - {kw}")
        else:
            print("   🤔 [분석 결과] 특별한 패턴은 감지되지 않았습니다. 다른 키워드를 추가해 보세요!")
            
    except StopIteration:
        print("\n✅ 모든 샘플 분석을 완료했습니다!")
        break
    except KeyError:
        print("\n❌ 오류: 'prompts' 필드를 찾을 수 없습니다. 데이터셋 구조를 확인해 주세요.")
        break

# ---------------------------------------------------------------------------
# 🌟 마무리 요약
# ---------------------------------------------------------------------------

print("\n==========================================================")
print("🎉 실습 완료! 정말 대단해요! ✨")
print("==========================================================")
print("🔥 배운 것 3가지:")
print("1. 스트리밍(Streaming) 데이터셋을 안전하게 다루는 방법 (try/except & take()).")
print("2. 텍스트 데이터(prompts)에서 의미 있는 패턴을 추출하는 로직을 설계하는 경험.")
print("3. 파이썬의 반복자(iterator) 패턴을 활용하여 효율적으로 데이터에 접근하는 방법.")
print("이 경험을 바탕으로, 다음엔 실제 이미지 생성 모델과 연동해 보세요!")